# Issue Triage Copilot — run and inspect

This notebook runs the full pipeline end-to-end: dataset → indexes → multi-agent triage → tracing → vanilla-vs-multi evaluation. Put `GITHUB_TOKEN` and `OPENAI_API_KEY` in `.env` (see README) before running.

Requirements: `pip install -e ".[dev]"` and `jupyter` / VS Code notebook support.

In [ ]:
import sys, os
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))

if Path(".env").exists():
    for line in Path(".env").read_text().splitlines():
        line = line.strip()
        if line and not line.startswith("#") and "=" in line:
            k, v = line.split("=", 1)
            os.environ.setdefault(k.strip(), v.strip())

print("OPENAI_API_KEY set:", bool(os.environ.get("OPENAI_API_KEY")))
print("GITHUB_TOKEN set:", bool(os.environ.get("GITHUB_TOKEN")))

## Step 1 — dataset (fetch or reuse)

If `triage/data/processed` already holds the persisted dataset, it is reused. Otherwise it fetches closed issues and process docs from GitHub, applies the held-out split (15%), and persists the corpus, held-out set, and docs.

In [ ]:
from triage.github import GitHubClient
from triage.fetch import fetch_issues, fetch_process_docs
from triage.rag.parse import parse_issue, parse_docs
from triage.evals.dataset import held_out_split
from triage.persist import save_records

processed = Path("triage/data/processed")
if not (processed / "issues_corpus.json").exists():
    token = os.environ.get("GITHUB_TOKEN")
    if not token:
        raise SystemExit("GITHUB_TOKEN missing — add it to .env")
    repos = ["scikit-learn/scikit-learn"]
    limit = 500
    records, docs = [], []
    with GitHubClient(token=token) as gh:
        for repo in repos:
            print(f"fetching {repo}...")
            records += [parse_issue(i) for i in fetch_issues(gh, repo, state="closed", limit=limit)]
            docs += parse_docs(fetch_process_docs(gh, repo))
    split = held_out_split(records)
    processed.mkdir(parents=True, exist_ok=True)
    save_records(split.corpus, processed / "issues_corpus.json")
    save_records(split.held_out, processed / "issues_held_out.json")
    save_records(docs, processed / "process_docs.json")
    print(f"corpus={len(split.corpus)} held_out={len(split.held_out)} docs={len(docs)}")
else:
    print("dataset already present — reusing it")

## Step 2 — build the indexes

Chunks the corpus issues (whole-issue signatures) and process docs (structural chunks), embeds them with `text-embedding-3-small` (disk-cached), and writes the Chroma indexes under `triage/data/indexes/`.

In [ ]:
from triage.persist import load_records
from triage.rag.parse import IssueRecord, ProcessDoc
from triage.rag.embed import Embedder
from triage.rag.index_build import build_issue_index, build_doc_index

indexes = Path("triage/data/indexes")
corpus = load_records(processed / "issues_corpus.json", IssueRecord)
docs = load_records(processed / "process_docs.json", ProcessDoc)
embedder = Embedder(cache_path=indexes / "embeddings.json")
issue_store = build_issue_index(corpus, embedder, indexes)
doc_store = build_doc_index(docs, embedder, indexes)
print(f"issue index: {issue_store.count()} chunks; doc index: {doc_store.count()} chunks")

## Step 3 — triage a held-out issue (multi-agent)

The orchestrator classifies the issue, fans out to the historical and process agents in parallel, then merges the evidence into a final decision with verified citations.

In [ ]:
import asyncio, json
from triage.mcp_tools.tools import TriageTools
from triage.mcp_tools.langchain import AgentToolbox
from triage.orchestration.graph import build_graph
from triage.orchestration.state import TriageState

held_out = load_records(processed / "issues_held_out.json", IssueRecord)
record = held_out[0]
query = f"{record.title}\n\n{record.body}"
issue_id = f"{record.repo}#{record.number}"
tools = TriageTools(indexes_dir=indexes, processed_dir=processed)

async def run():
    async with AgentToolbox(tools) as box:
        graph = build_graph(toolbox=box).compile()
        return await graph.ainvoke(TriageState(issue=query, issue_id=issue_id))

result = asyncio.run(run())
print(json.dumps(result["decision"].model_dump(mode="json"), indent=2))
print("\nneeds_human:", result["needs_human"])
print("actual labels:", record.labels, "| linked PRs:", record.linked_prs)

## Step 4 — tracing (latency per stage)

A local `Tracer` records per-node, per-LLM and per-tool latencies. `summary()` gives count / avg / p95 per stage; `save()` persists the raw events.

In [ ]:
from triage.tracing import Tracer

tracer = Tracer()

async def run_traced():
    async with AgentToolbox(tools) as box:
        graph = build_graph(toolbox=box, tracer=tracer).compile()
        return await graph.ainvoke(TriageState(issue=query, issue_id=issue_id))

result = asyncio.run(run_traced())
print(json.dumps(tracer.summary(), indent=2))
tracer.save(Path("triage/data/indexes/trace.json"))
print("trace events:", len(tracer.events()))

## Step 5 — vanilla RAG vs multi-agent comparison

Runs both systems over a small slice of the held-out set and prints the aggregate table (label accuracy, action overlap, judge scores, recall@k, p95 latency, cost).

In [ ]:
from triage.evals.runner import EvaluationRunner, format_results_table

runner = EvaluationRunner(indexes, processed)
results = runner.run_comparison(limit=3)
print(format_results_table(results))

## Wrap-up

- Inspect the raw trace at `triage/data/indexes/trace.json`.
- For a hosted observability backend (Langfuse / LangSmith), see the README's tracing section.
- Run the full evaluation over the whole held-out set with `python triage/scripts/run_evals.py`.